# Exploratory Data Analysis (EDA) on the Breast Cancer Wisconsin Dataset

**Exploratory Data Analysis (EDA)** is the process of investigating a dataset before, and sometimes during, machine learning development.

The purpose is not simply to make graphs. The purpose is to **understand what the data contains, discover relationships and patterns, detect problems, and form hypotheses that can later be tested with statistical or machine learning methods.**

The main questions EDA tries to answer:

```text
What data do I have?
What does each variable represent?
How are the values distributed?
How are variables related to each other?
How do variables relate to the target?
Are there unusual or problematic observations?
Are there patterns that could help prediction?
```

## How to use this notebook

Run the cells **top to bottom**. Every section follows the same pattern:

1. **Markdown**: what the technique is, why we use it, which questions it answers
2. **Code**: run it and look at the output
3. **Markdown**: how to read the result and how it connects to other techniques

## Table of contents

| # | Technique | Question it answers |
|---|-----------|--------------------|
| 1 | Dataset overview | What am I working with? |
| 2 | Target distribution | Are the classes balanced? |
| 3 | Missing value analysis | Is anything incomplete? |
| 4 | Statistical summary | What are the centers, spreads and ranges? |
| 5 | Histogram | What does one feature look like? |
| 6 | Density plot / KDE | What is the smooth shape of the distribution? |
| 7 | Box plot | Where are the median, quartiles and outliers? |
| 8 | Violin plot | How do distributions differ between groups? |
| 9 | Correlation analysis | Which variables move together? |
| 10 | Correlation matrix / heatmap | All pairwise correlations at once |
| 11 | Scatter plot | What does a relationship look like? |
| 12 | Pair plot | Many relationships at once |
| 13 | Feature vs target analysis | How does a feature differ between classes? |
| 14 | Outlier analysis | Which observations are unusual? |
| 15 | Class separation analysis | Can the classes be told apart? |
| 16 | PCA visualization | Is there structure in all 30 dimensions? |
| 17 | Duplicate analysis | Are there repeated rows (leakage risk)? |
| 18 | Feature importance analysis | What did a trained model rely on? |
| 19 | How everything connects | Worked examples, mental model and workflow |

# About the dataset

The data is the **Breast Cancer Wisconsin (Diagnostic)** dataset, the one you get from:

```python
from sklearn.datasets import load_breast_cancer
```

* **569 samples**, each one a breast mass that was examined with a fine needle aspirate (FNA).
* Each sample is described by **30 numeric features** computed from a digitized image of the cell nuclei in the sample.
* The **target** says whether the mass is **malignant** (cancerous) or **benign** (non-cancerous).

### The 30 features = 10 measurements x 3 statistics

For every nucleus measurement, three summary statistics are stored:

| Version | Meaning | Example column |
|---|---|---|
| `mean_...` | average over all nuclei in the image | `mean_radius` |
| `..._error` | standard error of that measurement | `radius_error` |
| `worst_...` | the "worst" (mean of the three largest values) | `worst_radius` |

The 10 base measurements:

| Measurement | What it describes |
|---|---|
| radius | average distance from the center to points on the nucleus boundary (size) |
| texture | variation in gray-scale values inside the nucleus |
| perimeter | length of the nucleus boundary (size) |
| area | area of the nucleus (size) |
| smoothness | local variation in radius lengths |
| compactness | perimeter² / area − 1 |
| concavity | severity of concave portions of the contour |
| concave points | number of concave portions of the contour |
| symmetry | how symmetric the nucleus is |
| fractal dimension | "coastline approximation" of the boundary (irregularity) |

### The target

In scikit-learn the target is encoded as `0 = malignant` and `1 = benign`.
In this notebook the benign class is called **"Healthy"**, and the classes are:

```text
Healthy      357
Malignant    212
```

We will create a readable `diagnosis` column (`Healthy` / `Malignant`) and also keep a numeric `target_num` column (`1 = Healthy`, `0 = Malignant`) so we can compute correlations with the target.

# 0. Load the data

The dataset has been exported to a CSV file. Load it with pandas.

In [ ]:
# Load the dataset from the CSV file
import pandas as pd
from sklearn.datasets import load_breast_cancer

raw_data = load_breast_cancer()
df = pd.DataFrame(raw_data.data, columns=raw_data.feature_names)
df['healthy_cells'] = raw_data.target

### (Optional) Look at the original scikit-learn description

The built-in loader ships with a full description of the dataset. This cell is only for reference. It is not needed for the rest of the notebook.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
print("Target names:", dict(enumerate(data.target_names)))
print()
print(data.DESCR)

---
# 1. Dataset Overview

## Purpose

The dataset overview is the first inspection of the data. Before analyzing relationships or distributions, you need to know **what you are actually working with**.

## Common checks

```python
df.shape
df.head()
df.info()
df.describe()
```

We run them one at a time so each output is visible.

### `df.shape` returns `(rows, columns)`

Expected here: 569 samples and 31 columns (30 features + 1 target).

In [ ]:
df.shape

### `df.head()`

Shows the first few records. Useful for understanding:

* What does one row look like?
* What do the columns contain?
* Are values represented as expected?
* Is the target included?

In [ ]:
df.head()

### `df.info()`

Shows column names, data types, number of non-null values and approximate memory usage.

This is especially useful for detecting numerical columns, categorical columns, missing values and unexpected data types. Here you should see 30 `float64` feature columns and one integer target column, all with 569 non-null values.

In [ ]:
df.info()

### `df.describe()`

Numerical summaries for each column: `count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`.

This gives you an initial idea of the **range, center, spread and potential extreme values**. (`.T` transposes the table so each feature is a row.)

In [ ]:
df.describe().T

---
# Preparation: imports, tidy names, helper columns

Before the deeper analysis, one setup cell:

1. Imports the plotting/analysis libraries.
2. Converts column names like `mean radius` to `mean_radius` (so they are easy to type; harmless if they already look like that).
3. **Detects the target column** and creates
   * `target_num`: `1 = Healthy`, `0 = Malignant` (numeric, used for correlations and models)
   * `diagnosis`: `"Healthy"` / `"Malignant"` (readable, used for plots)
4. Builds lists of feature names: `FEATURES` (all 30), `MEAN_FEATURES`, `ERROR_FEATURES`, `WORST_FEATURES`.

The last lines print the class counts. **You should see Healthy = 357 and Malignant = 212.** If the numbers are swapped, your CSV encodes the target the other way round. Just flip the mapping in the `diagnosis` line.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# 1) tidy column names: "mean radius" -> "mean_radius"
df.columns = df.columns.astype(str).str.strip().str.replace(" ", "_")

# 2) find the target column
candidates = ["target", "diagnosis", "label", "class"]
TARGET_COL = next((c for c in df.columns if c.lower() in candidates), df.columns[-1])

# 3) numeric target (sklearn convention: 0 = malignant, 1 = benign/"Healthy") + readable labels
raw_target = df[TARGET_COL]
if pd.api.types.is_numeric_dtype(raw_target):
    df["target_num"] = raw_target.astype(int)
else:  # text labels such as "M"/"B" or "Malignant"/"Benign"
    df["target_num"] = raw_target.astype(str).str.lower().str.startswith(("b", "h")).astype(int)
df["diagnosis"] = df["target_num"].map({0: "Malignant", 1: "Healthy"})

# 4) feature lists
NON_FEATURES = {TARGET_COL, "target_num", "diagnosis", "id"}
FEATURES = [c for c in df.select_dtypes(include="number").columns if c not in NON_FEATURES]
MEAN_FEATURES = [c for c in FEATURES if c.startswith("mean_")]
ERROR_FEATURES = [c for c in FEATURES if c.endswith("_error")]
WORST_FEATURES = [c for c in FEATURES if c.startswith("worst_")]

# consistent colors / order for all class-based plots
PALETTE = {"Healthy": "#2a9d8f", "Malignant": "#e76f51"}
ORDER = ["Healthy", "Malignant"]

# make sure the columns used later exist
required = ["mean_radius", "mean_area", "mean_perimeter", "mean_texture",
            "mean_concavity", "worst_radius", "worst_concave_points"]
missing_cols = [c for c in required if c not in df.columns]
assert not missing_cols, f"These columns were not found in your CSV: {missing_cols}. Columns are: {list(df.columns)}"

print(f"Target column detected : '{TARGET_COL}'")
print(f"Number of features     : {len(FEATURES)}")
print(f"Mean / Error / Worst   : {len(MEAN_FEATURES)} / {len(ERROR_FEATURES)} / {len(WORST_FEATURES)}")
print()
print(df["diagnosis"].value_counts())

---
# 2. Target Distribution

## Purpose

Understand the variable you are trying to predict.

For classification:

```python
df["target"].value_counts()
```

Usually visualized with a **bar chart**.

## Questions

* How many samples belong to each class?
* Is the dataset balanced?
* Is one class much smaller than another?
* Could class imbalance affect evaluation?

## Relationship with other analyses

Target distribution is important **before feature analysis**. Suppose you later observe that `mean_radius` has a different distribution for the two classes. That difference only makes sense when you already know how many samples belong to each class.

```text
Target Distribution
        ↓
provides context for
        ↓
Feature vs Target Analysis
```

In [ ]:
counts = df["diagnosis"].value_counts()
percent = df["diagnosis"].value_counts(normalize=True).mul(100).round(1)
pd.DataFrame({"count": counts, "percent": percent})

In [ ]:
ax = sns.countplot(data=df, x="diagnosis", order=ORDER, palette=PALETTE)
for container in ax.containers:
    ax.bar_label(container)
ax.set_title("Target distribution")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Number of samples")
plt.show()

In [ ]:
imbalance_ratio = counts.max() / counts.min()
majority_baseline = counts.max() / counts.sum()

print(f"Imbalance ratio (majority : minority) = {imbalance_ratio:.2f} : 1")
print(f"A model that ALWAYS predicts '{counts.idxmax()}' would already be {majority_baseline:.1%} accurate.")

### How to read it

The classes are somewhat imbalanced (about 63% / 37%), but not extremely.

The last cell shows why this matters for evaluation: a useless model that always predicts the majority class already scores about 63% **accuracy**. So a model should be judged against that baseline, and with metrics like precision / recall / F1, not accuracy alone.

---
# 3. Missing Value Analysis

## Purpose

Find incomplete observations.

```python
df.isnull().sum()
```

## Questions

* Which columns have missing values?
* How many values are missing?
* Are missing values concentrated in particular columns?
* Should they be removed or filled?

## Relationship with EDA

Missing-value analysis affects almost everything else:

```text
Missing Values
      ↓
Data Quality
      ↓
Distribution Analysis
      ↓
Relationship Analysis
```

You don't want to draw conclusions from a feature without knowing whether a significant portion of its data is missing.

In [ ]:
missing = df.isnull().sum()
print("Total missing values in the dataset:", missing.sum())
missing[missing > 0]

### This dataset is complete, so here is what a problem would look like

The real dataset has **no** missing values, so this technique looks "boring" here. To see how it works when there *is* a problem, the next cell makes a **copy** of the data and deliberately blanks out five `mean_area` values. Your original `df` is untouched.

In [ ]:
demo = df.copy()
demo.loc[[5, 40, 120, 300, 450], "mean_area"] = np.nan

report = pd.DataFrame({
    "missing": demo.isnull().sum(),
    "percent": demo.isnull().mean().mul(100).round(2),
})
report[report["missing"] > 0]

In [ ]:
# Two common ways to handle it (on the demo copy only)
dropped = demo.dropna(subset=["mean_area"])
filled = demo["mean_area"].fillna(demo["mean_area"].median())

print("Rows before:", len(demo))
print("Rows after dropping missing:", len(dropped))
print("Missing after median-fill:", filled.isnull().sum())

### How to read it

The report says `mean_area` has 5 missing values (~0.9%). A tiny fraction like that can usually be filled (for example with the median, which is robust to outliers) or the rows dropped. If a column were, say, 60% missing, you would think very differently about whether it is usable at all.

---
# 4. Statistical Summary

## Purpose

Get numerical information about individual variables.

```python
df.describe()
```

For a feature such as `mean_radius` you get:

| Statistic | What it tells you |
|---|---|
| **Mean** | typical average value |
| **Median** (50%) | middle observation |
| **Standard deviation** | how spread out the observations are |
| **Quartiles** (25%, 50%, 75%) | how the data is distributed across four sections |
| **Min / Max** | the observed range |

## Relationship with visualization

Statistical summaries and visualizations complement each other:

```text
df.describe()
      +
Histogram
      +
Box Plot
```

give you a much better understanding than any one of them alone. The statistics give you numbers. The plots let you see the structure behind those numbers.

In [ ]:
stats = df[FEATURES].describe().T
stats["skew"] = df[FEATURES].skew()
stats["range"] = stats["max"] - stats["min"]
stats.round(3)

In [ ]:
s = df["mean_radius"]
q1, q3 = s.quantile(0.25), s.quantile(0.75)

print("mean_radius")
print(f"  Mean   : {s.mean():.2f}")
print(f"  Median : {s.median():.2f}")
print(f"  Std    : {s.std():.2f}")
print(f"  Min/Max: {s.min():.2f} / {s.max():.2f}")
print(f"  Q1/Q3  : {q1:.2f} / {q3:.2f}   (IQR = {q3 - q1:.2f})")

In [ ]:
print("Smallest-scale features (by mean):")
print(stats["mean"].nsmallest(3).round(5))
print("\nLargest-scale features (by mean):")
print(stats["mean"].nlargest(3).round(1))

In [ ]:
# Which features are most skewed? (skew > 0 = long right tail, mean > median)
stats["skew"].sort_values(ascending=False).head(8).round(2)

### How to read it

* **Mean vs median:** when the mean is noticeably larger than the median, the distribution has a long right tail (positive skew). Many size-related features here (`area`, `perimeter`, `*_error`) are right-skewed.
* **Very different scales:** `mean_area` is in the hundreds, while `mean_smoothness` is around 0.1. This is a reason to **scale** features before distance-based methods and PCA (we will see the effect in Section 16).
* Numbers alone can hide structure. That is why the next sections add histograms and box plots.

---
# 5. Histogram

## Purpose

Understand the **distribution of one numerical variable**.

```python
df["mean_radius"].hist(bins=20)
```

A histogram divides values into ranges called **bins** and counts how many observations fall into each range.

## Questions

* What values are common? What values are rare?
* Where is the data concentrated?
* How wide is the distribution?
* Is it skewed?
* Are there multiple peaks?
* Are there possible outliers?

```text
Frequency
   |
   |          ███
   |        ███████
   |     ███████████
   |  ███████████████
   +--------------------> Value
```

## Relationship with other analyses

```text
Histogram    →  "What does this feature look like?"
Correlation  →  "Does this feature move with another feature?"
Scatter plot →  "What does that relationship actually look like?"
```

They answer different questions.

In [ ]:
df["mean_radius"].hist(bins=20, edgecolor="black")
plt.xlabel("mean_radius")
plt.ylabel("Frequency")
plt.title("Histogram of mean_radius (20 bins)")
plt.show()

### The number of bins changes the picture

Too few bins hide structure. Too many bins make the shape noisy. There is no single correct value, so try several.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, b in zip(axes, [5, 20, 60]):
    ax.hist(df["mean_radius"], bins=b, edgecolor="black")
    ax.set_title(f"bins = {b}")
    ax.set_xlabel("mean_radius")
axes[0].set_ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Add the mean and median as reference lines
plt.figure(figsize=(8, 4))
plt.hist(df["mean_radius"], bins=25, edgecolor="black", alpha=0.8)
plt.axvline(df["mean_radius"].mean(), color="red", linestyle="--", label="mean")
plt.axvline(df["mean_radius"].median(), color="green", linestyle="-", label="median")
plt.legend()
plt.title("mean_radius: mean vs median")
plt.show()

In [ ]:
# Look at all 10 "mean" features at once
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, col in zip(axes.ravel(), MEAN_FEATURES):
    sns.histplot(df[col], bins=25, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

### How to read it

* `mean_radius` is concentrated around a typical range, with a **right tail** of larger values (the mean sits slightly to the right of the median).
* Some features (e.g. `mean_texture`, `mean_smoothness`) look roughly bell-shaped. Others (e.g. `mean_area`, `mean_concavity`) are clearly skewed.
* A right tail is a hint: there may be **outliers**, or there may be **two groups mixed together**. We will test the "two groups" idea in the next section and again in Section 13.

---
# 6. Density Plot / KDE

## Purpose

Show the estimated smooth shape of a distribution.

```python
df["mean_radius"].plot(kind="kde")
```

Instead of discrete bars, KDE (kernel density estimation) produces a **smooth curve**.

## Questions

* Where are the high-density regions?
* Is the distribution unimodal or multimodal?
* Is it skewed?
* Are there multiple groups?

## Relationship with histogram

They study the **same concept** (distribution) but visualize it differently:

```text
Histogram → discrete bins
KDE       → smooth estimated density
```

They can therefore be used together.

In [ ]:
df["mean_radius"].plot(kind="kde", figsize=(8, 4))
plt.xlabel("mean_radius")
plt.title("KDE of mean_radius")
plt.show()

In [ ]:
# Histogram + KDE together (histogram on the density scale so the curve matches)
plt.figure(figsize=(8, 4))
sns.histplot(df["mean_radius"], bins=25, kde=True, stat="density")
plt.title("Histogram + KDE")
plt.show()

### KDE has a "bin-width-like" setting too

The smoothing amount (bandwidth) plays the role that the number of bins plays in a histogram. Too little smoothing gives a wiggly curve, too much smoothing hides real structure.

In [ ]:
plt.figure(figsize=(8, 4))
for bw in [0.3, 1, 3]:
    sns.kdeplot(df["mean_radius"], bw_adjust=bw, label=f"bw_adjust = {bw}")
plt.legend()
plt.title("Effect of KDE smoothing")
plt.show()

### "Are there multiple groups?" Split the KDE by class

The overall `mean_radius` distribution has a right tail. Let's check whether it is really **two different distributions mixed together**, one per diagnosis.

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(data=df, x="mean_radius", hue="diagnosis", palette=PALETTE,
            fill=True, common_norm=False, alpha=0.4)
plt.title("mean_radius KDE by diagnosis")
plt.show()

### How to read it

Separating by class reveals two bumps: **Healthy** samples centered on smaller radii and **Malignant** samples on larger radii, with some overlap. The right tail in the overall histogram was largely the malignant group. This is exactly the kind of "hidden group" a KDE (or a histogram split by group) can expose, and it is our first hint that `mean_radius` is useful for classification.

---
# 7. Box Plot

## Purpose

Summarize a distribution compactly and identify potential outliers.

A box plot shows:

```text
Minimum / lower range (lower whisker)
Q1   (25th percentile, bottom of the box)
Median
Q3   (75th percentile, top of the box)
Upper range (upper whisker)
Potential outliers (individual points)
```

## Questions

* Where is the median?
* How spread out is the central portion?
* Are there extreme observations?
* Is the distribution asymmetric?

## Relationship with histogram

```text
Histogram → shows the overall shape
Box Plot  → shows median, quartiles, spread and outliers compactly
```

Together they give both the **shape** and the **statistical summary** of the distribution.

In [ ]:
plt.figure(figsize=(8, 2.5))
sns.boxplot(x=df["mean_radius"])
plt.title("Box plot of mean_radius")
plt.show()

In [ ]:
# Histogram + box plot sharing the same x-axis: shape AND summary in one view
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9, 6),
                               gridspec_kw={"height_ratios": [3, 1]})
sns.histplot(df["mean_radius"], bins=25, kde=True, ax=ax1)
sns.boxplot(x=df["mean_radius"], ax=ax2)
ax1.set_title("mean_radius: histogram (shape) + box plot (summary)")
plt.tight_layout()
plt.show()

In [ ]:
# What the box plot is actually computing (Tukey's rule with whiskers at 1.5 x IQR)
s = df["mean_radius"]
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")
print(f"Whisker limits: [{lower:.2f}, {upper:.2f}]")
print(f"Points outside the whiskers: {((s < lower) | (s > upper)).sum()}")

In [ ]:
# Compare all "mean" features on one plot by standardizing them (z-scores)
z = (df[MEAN_FEATURES] - df[MEAN_FEATURES].mean()) / df[MEAN_FEATURES].std()
plt.figure(figsize=(9, 5))
sns.boxplot(data=z, orient="h")
plt.title("Box plots of standardized mean features")
plt.xlabel("standard deviations from the mean")
plt.show()

### How to read it

* The box is the middle 50% of the data. The line inside is the median. If the line is off-center in the box, the distribution is asymmetric.
* The dots beyond the whiskers are **potential outliers**. Note that they are only *potential*. In Section 14 we look at whether they are errors or real cases.
* Standardizing puts features with very different scales on the same axis, which makes the "which features have the most extreme points?" question answerable at a glance.

---
# 8. Violin Plot

## Purpose

Compare distributions **between groups** while also showing their shape.

```python
sns.violinplot(x="target", y="mean_radius", data=df)
```

```text
Healthy       Malignant

  /\             /\
 /  \           /  \
|    |         |    |
 \  /           \  /
  \/             \/
```

## Questions

* Where are the values concentrated?
* How different are the classes?
* Do their distributions overlap?
* Are there multiple density regions?

## Relationship with box plot

A violin plot can be thought of as:

```text
Distribution Shape (a mirrored KDE)
        +
Box-plot-like summary (inside)
```

So it is especially useful for **feature vs target analysis**.

In [ ]:
plt.figure(figsize=(6, 5))
sns.violinplot(x="diagnosis", y="mean_radius", data=df, order=ORDER, palette=PALETTE)
plt.title("mean_radius by diagnosis")
plt.show()

In [ ]:
# The same with the box-plot summary drawn inside each violin
plt.figure(figsize=(6, 5))
sns.violinplot(x="diagnosis", y="mean_radius", data=df, order=ORDER,
               palette=PALETTE, inner="box")
plt.title("mean_radius by diagnosis (with box inside)")
plt.show()

In [ ]:
# Four different features side by side
cols = ["mean_radius", "mean_texture", "mean_concavity", "worst_concave_points"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, col in zip(axes, cols):
    sns.violinplot(x="diagnosis", y=col, data=df, order=ORDER, palette=PALETTE, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

### How to read it

* **`mean_radius`, `mean_concavity`, `worst_concave_points`**: the two violins sit at clearly different heights with limited overlap. These features should help a classifier.
* **`mean_texture`**: the violins overlap heavily, so this feature separates the classes much less.
* Compare the widths too. A wide part of the violin means many samples there.

---
# 9. Correlation Analysis

## Purpose

Measure the degree to which numerical variables move together, usually using a specific correlation coefficient such as **Pearson correlation**.

```text
corr(radius, area) = 0.98
```

means they have a very strong positive **linear** association.

```python
df.corr()
```

## Questions

* Which numerical variables move together?
* Which variables have strong positive relationships?
* Which have strong negative relationships?
* Which variables appear weakly linearly related?
* Which variables may be redundant?

## Example

```text
Radius ↔ Area       = 0.98
Area ↔ Perimeter    = 0.99
```

You might hypothesize:

> Radius, area, and perimeter are largely describing the same underlying concept: **nucleus size**.

## Important

Correlation is **not the same as causation**. And correlation mainly describes a particular form of association (linear association for Pearson).

A correlation near zero does **not** necessarily mean "no relationship exists". There may be a nonlinear relationship that Pearson correlation does not capture well.

In [ ]:
# Correlation between a few specific pairs
pairs = [("mean_radius", "mean_area"),
         ("mean_area", "mean_perimeter"),
         ("mean_radius", "mean_perimeter"),
         ("mean_radius", "mean_texture")]

for a, b in pairs:
    print(f"corr({a}, {b}) = {df[a].corr(df[b]):.3f}")

In [ ]:
# The full correlation table (30 x 30). Here just the first 6 columns as a preview.
corr = df[FEATURES].corr()
corr.iloc[:8, :6].round(2)

### Demonstration: "zero correlation" does not mean "no relationship"

Here `y` is **perfectly determined** by `x` (`y = x²`). Pearson correlation still comes out near zero because the relationship is curved and symmetric, not linear.

In [ ]:
x = np.linspace(-3, 3, 200)
y = x ** 2

print(f"Pearson correlation between x and x^2 = {np.corrcoef(x, y)[0, 1]:.3f}")

plt.figure(figsize=(5, 4))
plt.scatter(x, y, s=10)
plt.title("A perfect relationship with ~0 Pearson correlation")
plt.show()

### How to read it

* `radius`, `perimeter` and `area` have correlations of roughly 0.98 to 1.0. They are close to **redundant**: they mostly measure the same thing (size).
* `mean_radius` and `mean_texture` are only weakly correlated (~0.3): different information.
* The `x²` example is the reason correlation should always be **paired with a scatter plot** (Section 11).

---
# 10. Correlation Matrix / Heatmap

## Purpose

Look at many pairwise correlations at once.

```python
corr = df.corr()
sns.heatmap(corr)
```

Instead of checking `radius ↔ area`, `radius ↔ perimeter`, `radius ↔ texture`, ... individually, the matrix displays them all together.

## Questions

* Which variables are strongly related?
* Which variables are negatively related?
* Which groups of variables appear redundant?
* Which variables have a strong relationship with the target?

## Relationship with relationship analysis

```text
Relationship Analysis     ← broad objective
        ↑
Correlation Analysis      ← one statistical method
```

The correlation matrix gives you **evidence** about relationships. You then investigate those relationships further using plots and domain knowledge.

In [ ]:
plt.figure(figsize=(14, 11))
sns.heatmap(df[FEATURES].corr(), cmap="coolwarm", vmin=-1, vmax=1, center=0,
            square=True, cbar_kws={"shrink": 0.7})
plt.title("Correlation heatmap: all 30 features")
plt.show()

In [ ]:
# A smaller, annotated version for the 10 "mean" features
plt.figure(figsize=(10, 8))
sns.heatmap(df[MEAN_FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, square=True)
plt.title("Correlation heatmap: mean features")
plt.show()

In [ ]:
# List the most strongly correlated (potentially redundant) feature pairs
corr = df[FEATURES].corr()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))  # keep each pair once
pairs = upper.stack().rename("corr").reset_index()
pairs.columns = ["feature_1", "feature_2", "corr"]

strong = pairs[pairs["corr"].abs() > 0.9].sort_values("corr", ascending=False)
print(f"{len(strong)} feature pairs have |correlation| > 0.9 (out of {len(pairs)} pairs)")
strong.head(15).round(3)

### How to read it

* **Red blocks** (positive) form clusters of features that move together: the size-related features (`radius`, `perimeter`, `area`, in their mean and worst versions) and the shape-related ones (`concavity`, `concave_points`, `compactness`).
* The long list of |corr| > 0.9 pairs shows the dataset contains a lot of **redundancy**. Many of the 30 features carry overlapping information. This matters for interpretation (Section 18) and is why PCA (Section 16) can compress the data so well.
* The heatmap gives you leads. It does not give you conclusions. Follow up with scatter plots.

---
# 11. Scatter Plot

## Purpose

Visually investigate the relationship between **two numerical variables**.

```python
plt.scatter(df["mean_radius"], df["mean_area"])
```

## Questions

* Is the relationship positive? Negative?
* Is it approximately linear? Nonlinear?
* Are there clusters?
* Are there outliers?
* Does the relationship change across different regions?

## Why use it with correlation?

Suppose `corr(radius, area) = 0.98`. The correlation gives you a **numerical summary**. The scatter plot lets you **see why** the correlation is high:

```text
      •
    •
   •
  •
 •
```

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(df["mean_radius"], df["mean_area"], alpha=0.6)
plt.xlabel("mean_radius")
plt.ylabel("mean_area")
r = df["mean_radius"].corr(df["mean_area"])
plt.title(f"mean_radius vs mean_area (corr = {r:.3f})")
plt.show()

In [ ]:
# Contrast: a weak relationship
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (a, b) in zip(axes, [("mean_radius", "mean_area"), ("mean_radius", "mean_texture")]):
    ax.scatter(df[a], df[b], alpha=0.6)
    slope, intercept = np.polyfit(df[a], df[b], 1)
    xs = np.linspace(df[a].min(), df[a].max(), 100)
    ax.plot(xs, slope * xs + intercept, color="red")
    ax.set_xlabel(a)
    ax.set_ylabel(b)
    ax.set_title(f"corr = {df[a].corr(df[b]):.2f}")

plt.tight_layout()
plt.show()

In [ ]:
# Add the diagnosis as color: a first look at whether the classes separate
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="mean_radius", y="mean_texture", hue="diagnosis",
                palette=PALETTE, alpha=0.7)
plt.title("mean_radius vs mean_texture, colored by diagnosis")
plt.show()

### How to read it

* Left plot: points hug a line. That is what a correlation of ~0.99 *looks like*. (The area grows with the square of the radius, so you can see a very slight curve.)
* Right plot: a shapeless cloud. That is what a correlation of ~0.3 looks like.
* The colored plot shows the **same two features** and the classes are already fairly separable along the radius axis but mixed along texture. Scatter plots contribute to relationship analysis, outlier analysis *and* class separation.

---
# 12. Pair Plot

## Purpose

Perform many two-variable visualizations **simultaneously**.

For features `radius`, `area`, `texture`, `concavity` a pair plot shows every pair (`radius ↔ area`, `radius ↔ texture`, ... ) plus the distribution of each variable on the diagonal.

## Questions

* Which variables appear related?
* Which relationships are nonlinear?
* Are there clusters?
* Are classes visually separable?
* Are there obvious outliers?

## Relationship with scatter plot

```text
Scatter Plot → investigate ONE relationship
Pair Plot    → investigate MANY relationships at once
```

Pair plots grow quickly (N features → N² panels), so use a **handful** of features chosen from earlier steps, not all 30.

In [ ]:
# May take a few seconds
cols = ["mean_radius", "mean_area", "mean_texture", "mean_concavity"]
g = sns.pairplot(df, vars=cols, hue="diagnosis", palette=PALETTE,
                 diag_kind="kde", corner=True, plot_kws={"alpha": 0.6, "s": 20})
g.figure.suptitle("Pair plot of selected mean features", y=1.02)
plt.show()

### How to read it

* **Diagonal**: the distribution of each feature by class (a KDE version of Section 6).
* **`mean_radius` vs `mean_area`**: strong curved line (redundant features).
* **Any panel involving `mean_concavity` or `mean_radius`**: the two colors form partly different regions, so the classes are separable.
* **Panels with `mean_texture`**: the colors overlap much more.

---
# 13. Feature vs Target Analysis

## Purpose

Investigate how a feature **differs according to the target**.

```text
mean radius
        ↓
Healthy vs Malignant
```

```python
sns.boxplot(x="target", y="mean_radius", data=df)
```

## Questions

* Do the classes have different distributions?
* Is there substantial overlap?
* Does the feature appear useful for classification?
* Are differences consistent or driven by a few observations?

## Relationship with correlation

```text
Correlation        → numerical summary of association with the target
Box / Violin plot  → distribution of the feature WITHIN each class
```

Together they give more context.

> **Reading the sign:** `target_num` is `1 = Healthy`, `0 = Malignant`. So a **negative** correlation means "larger values → more likely malignant".

In [ ]:
# Correlation of every feature with the (numeric) target
target_corr = df[FEATURES].corrwith(df["target_num"]).sort_values()

plt.figure(figsize=(8, 9))
colors = [PALETTE["Malignant"] if v < 0 else PALETTE["Healthy"] for v in target_corr]
target_corr.plot(kind="barh", color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlation of each feature with target (1 = Healthy)")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

In [ ]:
top_features = target_corr.abs().sort_values(ascending=False).head(4).index.tolist()
print("Strongest 4 features by |correlation| with the target:")
for f in top_features:
    print(f"  {f:<25} {target_corr[f]:+.3f}")

In [ ]:
# Box plots: how does each top feature differ between classes?
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, col in zip(axes, top_features):
    sns.boxplot(x="diagnosis", y=col, data=df, order=ORDER, palette=PALETTE, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Overlapping histograms: the same idea, showing the full shape and the overlap
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, top_features):
    sns.histplot(data=df, x=col, hue="diagnosis", palette=PALETTE,
                 bins=30, element="step", ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# The group means side by side
df.groupby("diagnosis")[top_features].mean().T.round(3)

### How to read it

* The correlation bar chart ranks features by how strongly they are linearly associated with the diagnosis. Features like `worst_concave_points`, `worst_perimeter`, `mean_concave_points`, `worst_radius` and `mean_radius` are near the top (about −0.7 to −0.8). Features like `mean_fractal_dimension` and `texture_error` are near zero.
* The box plots and histograms show **how** the classes differ: the malignant group is shifted to larger values, and the overlap region tells you how many samples are "ambiguous" on that feature alone.
* If the box plot for a feature showed nearly identical boxes, that feature would probably not help much on its own.

---
# 14. Outlier Analysis

## Purpose

Identify observations that are unusually far from the rest of the data.

## Common tools

* Box plots
* IQR method
* Z-score
* Histograms
* Scatter plots

## Questions

* Is this observation genuinely unusual?
* Is it a data-entry problem?
* Is it a rare but valid observation?
* Could it represent an important case?

## Relationship with other analysis

Outlier analysis is not isolated. An outlier found in a histogram can be investigated with:

```text
Histogram
    ↓
Possible Outlier
    ↓
Scatter Plot
    ↓
Check Relationships
    ↓
Domain Knowledge
    ↓
Decide whether it is meaningful
```

**You should not automatically delete every outlier.**

In [ ]:
def iqr_outliers(s, k=1.5):
    """Boolean mask: True where a value lies outside [Q1 - k*IQR, Q3 + k*IQR]."""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)

iqr_counts = pd.Series({c: int(iqr_outliers(df[c]).sum()) for c in FEATURES})

z_scores = (df[FEATURES] - df[FEATURES].mean()) / df[FEATURES].std()
z_counts = (z_scores.abs() > 3).sum()

outlier_table = pd.DataFrame({"IQR method (1.5 x IQR)": iqr_counts, "Z-score method (|z| > 3)": z_counts})
outlier_table.sort_values("IQR method (1.5 x IQR)", ascending=False).head(10)

The two methods give different counts. That is normal. The IQR rule is based on quartiles (robust to skew), while the z-score assumes the data is roughly symmetric and uses the mean and standard deviation (which the extreme values themselves distort). Neither is "the truth". They are two ways to flag candidates for a closer look.

### Follow the chain: histogram → possible outlier → scatter plot → decide

Take `mean_area`, one of the most skewed features.

In [ ]:
mask = iqr_outliers(df["mean_area"])
print(f"{mask.sum()} samples flagged as outliers in mean_area\n")

# Step 1: the histogram shows a long right tail
# Step 2: the scatter plot shows whether those points fit the general pattern
plt.figure(figsize=(7, 5))
plt.scatter(df.loc[~mask, "mean_radius"], df.loc[~mask, "mean_area"], alpha=0.5, label="normal")
plt.scatter(df.loc[mask, "mean_radius"], df.loc[mask, "mean_area"],
            color="red", edgecolor="black", label="flagged outlier")
plt.xlabel("mean_radius")
plt.ylabel("mean_area")
plt.legend()
plt.title("Flagged outliers still follow the radius-area pattern")
plt.show()

In [ ]:
# Step 3: do the flagged samples belong to a particular class?
print("Diagnosis of the flagged samples:")
print(df.loc[mask, "diagnosis"].value_counts())
print("\nDiagnosis in the whole dataset:")
print(df["diagnosis"].value_counts())

### How to read it

* The flagged points are **not errors**. They lie right along the same radius-area curve as everyone else, just further out. They are simply large nuclei.
* They are also concentrated in the **Malignant** class. These "outliers" are among the most informative cases in the dataset. Deleting them would remove exactly the samples that carry the signal.
* The lesson: flag → investigate → use domain knowledge → *then* decide. "Statistically unusual" is not the same as "wrong".

---
# 15. Class Separation Analysis

## Purpose

Investigate whether the classes appear **distinguishable** based on the available measurements.

`Healthy` and `Malignant` might form visibly different regions in a scatter plot.

## Tools

* Scatter plots
* Pair plots
* Box plots
* Violin plots
* PCA visualization

## Questions

* Do classes occupy different regions?
* How much do they overlap?
* Are there obvious boundaries?
* Which features appear to separate the classes?

## Relationship with feature vs target analysis

```text
Feature vs Target → study ONE feature's behavior across classes
Class Separation  → study whether the classes can be distinguished using one or MORE measurements
```

In [ ]:
plt.figure(figsize=(7, 5.5))
sns.scatterplot(data=df, x="worst_radius", y="worst_concave_points", hue="diagnosis",
                palette=PALETTE, alpha=0.75)
plt.title("Two features, two fairly distinct regions")
plt.show()

### Put a number on "how separable is this feature?"

Two simple measures for each feature:

* **Separation AUC**: if you picked one Healthy and one Malignant sample at random, how often does this single feature rank them correctly? `0.5` = no better than a coin flip, `1.0` = perfect. (We use `max(auc, 1 - auc)` so the direction doesn't matter.)
* **Cohen's d**: the difference between class means measured in pooled standard deviations. Larger |d| = the class centers are further apart relative to their spread.

In [ ]:
from sklearn.metrics import roc_auc_score

def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na - 1) * a.var() + (nb - 1) * b.var()) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled

healthy = df[df["diagnosis"] == "Healthy"]
malignant = df[df["diagnosis"] == "Malignant"]

sep = pd.DataFrame(index=FEATURES)
auc = pd.Series({c: roc_auc_score(df["target_num"], df[c]) for c in FEATURES})
sep["separation_auc"] = np.maximum(auc, 1 - auc)
sep["cohens_d"] = pd.Series({c: cohens_d(malignant[c], healthy[c]) for c in FEATURES})
sep = sep.sort_values("separation_auc", ascending=False)

sep.head(10).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
sep["separation_auc"].sort_values().plot(kind="barh", ax=ax)
ax.axvline(0.5, color="red", linestyle="--", label="coin flip")
ax.set_xlim(0.4, 1.0)
ax.set_title("How well does each SINGLE feature separate the classes?")
ax.set_xlabel("separation AUC")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Contrast: the two WEAKEST separators
weak = sep["separation_auc"].nsmallest(2).index.tolist()
print("Weakest separators:", weak)

plt.figure(figsize=(7, 5.5))
sns.scatterplot(data=df, x=weak[0], y=weak[1], hue="diagnosis", palette=PALETTE, alpha=0.75)
plt.title("Two weak features: the classes are mixed together")
plt.show()

### How to read it

* Using just two well-chosen features (`worst_radius`, `worst_concave_points`) the classes already occupy different regions with a modest overlap zone near the boundary.
* Several single features reach a separation AUC of 0.95 or higher. This dataset is known to be "easy" for classification.
* Two weak features form one mixed cloud. The difference between this plot and the previous scatter is what "class separation" means visually.
* This analysis points to a **hypothesis**: "a classifier should be able to reach high accuracy". Section 18 tests it.

---
# 16. PCA Visualization

## Purpose

PCA (Principal Component Analysis) transforms many numerical variables into a smaller number of **new dimensions**.

```text
30 features
      ↓
PCA
      ↓
2 principal components
      ↓
2D visualization
```

Then you can plot `PC1 vs PC2`.

## Questions

* Does the dataset have large-scale structure?
* Do samples form clusters?
* Do classes appear separated?
* Is much of the variation represented by a few components?

## Important

PCA does **not** simply mean *"choose the two most important original features"*. The principal components are **combinations** of the original variables.

Also: PCA is driven by variance, so features must be **standardized** first. Otherwise features with big numbers (like `area`) dominate.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_scaled = StandardScaler().fit_transform(df[FEATURES])
pca = PCA().fit(X_scaled)

evr = pca.explained_variance_ratio_
cumulative = np.cumsum(evr)

print("Variance explained by the first 5 components:", np.round(evr[:5], 3))
print(f"PC1 + PC2 together: {cumulative[1]:.1%}")
print(f"Components needed for 90% of the variance: {np.argmax(cumulative >= 0.90) + 1}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].bar(range(1, 11), evr[:10])
ax[0].set_title("Variance explained by each component (first 10)")
ax[0].set_xlabel("component")

ax[1].plot(range(1, len(cumulative) + 1), cumulative, marker="o")
ax[1].axhline(0.90, color="red", linestyle="--")
ax[1].set_title("Cumulative variance explained")
ax[1].set_xlabel("number of components")

plt.tight_layout()
plt.show()

In [ ]:
pcs = pca.transform(X_scaled)[:, :2]
pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"])
pca_df["diagnosis"] = df["diagnosis"].values

plt.figure(figsize=(7, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="diagnosis", palette=PALETTE, alpha=0.75)
plt.title(f"PCA: 30 features -> 2 components ({cumulative[1]:.0%} of the variance)")
plt.show()

In [ ]:
# What ARE the components? Each is a weighted combination of the original features.
loadings = pd.DataFrame(pca.components_[:2].T, index=FEATURES, columns=["PC1", "PC2"])
order = loadings["PC1"].abs().sort_values(ascending=False).index
loadings.loc[order].head(10).round(3)

In [ ]:
# Why scaling matters: PCA on the RAW (unscaled) features
raw_pca = PCA().fit(df[FEATURES])
print("Variance explained by PC1, PC2, PC3 WITHOUT scaling:", np.round(raw_pca.explained_variance_ratio_[:3], 3))
print("Variance explained by PC1, PC2, PC3 WITH scaling   :", np.round(evr[:3], 3))

### How to read it

* Just **two components** capture roughly 60+% of the variation, and about 10 components capture ~95%. This is the mathematical face of the redundancy we saw in the correlation heatmap.
* In the PC1 vs PC2 plot, the two diagnoses form two largely separate clouds, even though **the diagnosis was never given to PCA**. The separation comes purely from the structure in the measurements. Strong evidence that the classes are separable.
* The loadings table shows PC1 is a blend of many features (mostly size and concavity-related, all with the same sign). It is not "just radius".
* Without scaling, PC1 would swallow nearly all variance because `area` has huge numbers. It would be measuring units, not structure.

---
# 17. Duplicate Analysis

## Purpose

Find repeated observations.

```python
df.duplicated().sum()
```

## Questions

* Are there duplicate records?
* Could repeated samples cause problems?
* Could the same observation appear in both training and testing data?

## Why it matters

Suppose the exact same sample appears in:

```text
Training Set
+
Testing Set
```

The model may appear to perform extremely well because it has effectively **already seen the test observation**. So duplicate analysis is also related to **data leakage prevention**.

In [ ]:
n_dup = df.duplicated(subset=FEATURES).sum()
print("Duplicate rows in the dataset:", n_dup)

### Demonstration: what leakage looks like

The dataset has no duplicates, so we create a **copy** with 60 rows accidentally repeated, split it randomly, and count how many test rows also exist in the training set.

In [ ]:
from sklearn.model_selection import train_test_split

leaky = pd.concat([df, df.sample(60, random_state=1)], ignore_index=True)
print("Duplicates in the corrupted copy:", leaky.duplicated(subset=FEATURES).sum())

train_l, test_l = train_test_split(leaky, test_size=0.25, random_state=42)
overlap = test_l.merge(train_l[FEATURES].drop_duplicates(), on=FEATURES, how="inner")
print(f"{len(overlap)} of {len(test_l)} test rows also appear in the training set")

# The fix is one line:
clean = leaky.drop_duplicates(subset=FEATURES)
print("Rows after drop_duplicates():", len(clean), "(original had", len(df), ")")

### How to read it

The real dataset is clean (0 duplicates), so nothing needs fixing. In the demo, a chunk of the test set is literally also in the training set. A model could just *memorize* those rows and look better than it really is. Always check for duplicates **before** splitting into train and test.

---
# 18. Feature Importance Analysis

## Purpose

After training a model, investigate which variables the model relied on **according to the particular importance method being used**.

```python
model.feature_importances_
```

## Questions

* Which features contributed strongly according to this model/method?
* Which features contributed less?
* Does the model's behavior resemble what EDA suggested?

## Important distinction

This is different from ordinary EDA because it usually happens **after a model exists**.

```text
EDA    → "What does the data appear to say?"
Model  → "What did this trained model use?"
```

You can then compare the two. That provides a useful consistency check, although feature importance should **not** automatically be interpreted as causation.

We use a Random Forest as a baseline and two importance methods:

* **Impurity-based** (`feature_importances_`): computed during training. Can favor features with many distinct values.
* **Permutation importance**: shuffle one feature on the *test* set and see how much the score drops.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.inspection import permutation_importance

X = df[FEATURES]
y = df["target_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, pred):.3f}   (majority-class baseline was {majority_baseline:.3f})\n")
print(classification_report(y_test, pred, target_names=["Malignant", "Healthy"]))

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
imp.head(10).sort_values().plot(kind="barh")
plt.title("Random Forest feature importance (top 10)")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

In [ ]:
perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
perm_imp.head(10).sort_values().plot(kind="barh", color="#e9c46a")
plt.title("Permutation importance on the test set (top 10)")
plt.xlabel("drop in accuracy when shuffled")
plt.tight_layout()
plt.show()

### Compare what the model used with what EDA suggested

We rank all features three ways: by |correlation| with the target (EDA, Section 13), by Random Forest importance, and by permutation importance. Rank 1 = most important.

In [ ]:
comparison = pd.DataFrame({
    "EDA |corr| with target": df[FEATURES].corrwith(df["target_num"]).abs(),
    "RF importance": imp,
    "Permutation importance": perm_imp,
})

ranks = comparison.rank(ascending=False).astype(int)
ranks.columns = ["EDA rank", "RF rank", "Permutation rank"]

print("Top 10 features according to the Random Forest, with their ranks in each method:")
display(ranks.sort_values("RF rank").head(10))

print("How similar are the three rankings? (Spearman rank correlation)")
display(ranks.corr(method="spearman").round(2))

### How to read it

* **Consistency check:** the features that EDA flagged (Section 13: `worst_concave_points`, `worst_perimeter`, `worst_radius`, `mean_concave_points`, ...) also show up near the top of the model's importance lists. EDA and the model tell a similar story.
* **But not identical:** the model's ranking is not exactly the EDA ranking. Correlated features (radius / perimeter / area, from Section 10) **share** importance, so any one of them can look less important than it really is. Permutation importance is affected even more: shuffling `worst_area` barely hurts if `worst_radius` and `worst_perimeter` still carry the same information, so a strongly informative but redundant feature can get a low score (check the low Spearman values in the table above).
* **Not causation:** importance says "the model used it", not "it causes malignancy".
* The high test accuracy (~95%+) confirms the **hypothesis** from Sections 15 and 16: the classes are highly separable with these measurements.

---
# 19. How These Analyses Connect

This is the most important part. **EDA is not a collection of unrelated graphs.** You can think of it as a chain of questions.

```text
                    DATASET
                       |
                       v
              Dataset Overview
                       |
          +------------+------------+
          |                         |
          v                         v
     Data Quality              Target
          |                         |
          v                         v
 Missing Values               Class Balance
 Duplicates                       |
          |                         |
          +------------+------------+
                       |
                       v
              Individual Features
                       |
             +---------+---------+
             |                   |
             v                   v
        Distribution       Statistics
             |                   |
       +-----+-----+             |
       |           |             |
       v           v             |
 Histogram       Box Plot        |
       |           |             |
       +-----+-----+-------------+
             |
             v
       Feature Relationships
             |
       +-----+-----+
       |           |
       v           v
 Correlation   Scatter Plot
       |           |
       +-----+-----+
             |
             v
       Relationship Analysis
             |
             v
      Feature ↔ Target
             |
       +-----+-----+
       |           |
       v           v
   Box/Violin   Scatter/Pairs
       |           |
       +-----+-----+
             |
             v
       Class Separation
             |
             v
        Model Training
             |
             v
      Feature Importance
             |
             v
      Compare Model
      With EDA Findings
```

## The difference between "Analysis" and "Tools"

**Analysis = the question / goal**

```text
Distribution Analysis
Relationship Analysis
Outlier Analysis
Class Separation Analysis
Feature Importance Analysis
```

**Tools = how you investigate it**

```text
Histogram
KDE
Correlation
Scatter Plot
Box Plot
Violin Plot
PCA
Statistical Tests
```

One analysis can use several tools. One tool can also contribute to several analyses. For example, a scatter plot serves relationship analysis, outlier analysis and class separation:

```text
                    Scatter Plot
                   /      |       \
                  /       |        \
                 v        v         v
          Relationship  Outlier   Class
            Analysis    Analysis  Separation
```

There is **not** a strict one-to-one relationship between an analysis and a visualization. (We saw this already: the scatter plot appeared in Sections 11, 14 and 15.)

## A better mental model

Instead of memorizing `Histogram = EDA`, `Correlation = EDA`, `Scatter = EDA`, think:

```text
                 EDA
                  |
        "What is happening in my data?"
                  |
       +----------+----------+
       |          |          |
       v          v          v
 Distribution  Relationships  Quality
       |          |          |
       v          v          v
 Histogram    Correlation   Missing Values
 KDE          Scatter       Duplicates
 Box Plot     Pair Plot     Outliers
              Box Plot
                  |
                  v
             Target Analysis
                  |
                  v
            Class Separation
                  |
                  v
             Model Building
                  |
                  v
          Model Interpretation
```

---
# Worked example 1: Relationship analysis (size features)

Suppose you start with `mean_radius`, `mean_area` and `mean_perimeter`.

**Step 1: Distribution.** A histogram of `mean_radius` shows most observations concentrated in a particular range, with some larger values.

**Step 2: Correlation.** `corr(radius, area) ≈ 0.98`: evidence that they move together strongly.

**Step 3: Scatter plot.** `radius vs area` shows a strong upward pattern, so the numerical correlation now has a visual explanation.

**Step 4: Domain interpretation.** Radius → size. Area → size. Therefore radius and area are strongly related because both largely describe **nucleus size**.

```text
Correlation
      +
Scatter Plot
      +
Domain Knowledge
      ↓
Relationship Interpretation
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Step 1 - distribution
sns.histplot(df["mean_radius"], bins=25, kde=True, ax=axes[0])
axes[0].set_title("Step 1 - Distribution of mean_radius")

# Step 3 - scatter
axes[1].scatter(df["mean_radius"], df["mean_area"], alpha=0.6)
axes[1].set_xlabel("mean_radius")
axes[1].set_ylabel("mean_area")
axes[1].set_title("Step 3 - radius vs area")

plt.tight_layout()
plt.show()

# Step 2 - correlation
print("Step 2 - correlations among the size features:")
df[["mean_radius", "mean_perimeter", "mean_area"]].corr().round(3)

---
# Worked example 2: Feature vs target (`worst_concave_points`)

Suppose you find `corr(worst_concave_points, target) ≈ -0.79` (Section 13). You now investigate further:

* **Histogram**: what values are common?
* **Box plot**: compare Healthy vs Malignant
* **Violin plot**: the full distribution for each class
* **Scatter plot**: compare it with another related feature, e.g. `worst_concave_points` vs `worst_radius`. Now you can form a stronger conclusion about the role of **boundary irregularity**.

In [ ]:
feat = "worst_concave_points"
print(f"corr({feat}, target) = {df[feat].corr(df['target_num']):.3f}")

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

sns.histplot(df[feat], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Histogram: overall distribution")

sns.boxplot(x="diagnosis", y=feat, data=df, order=ORDER, palette=PALETTE, ax=axes[1])
axes[1].set_title("Box plot: by class")

sns.violinplot(x="diagnosis", y=feat, data=df, order=ORDER, palette=PALETTE, ax=axes[2])
axes[2].set_title("Violin plot: by class")

sns.scatterplot(data=df, x="worst_radius", y=feat, hue="diagnosis", palette=PALETTE,
                alpha=0.75, ax=axes[3])
axes[3].set_title("Scatter: vs worst_radius")

plt.tight_layout()
plt.show()

### Conclusion from example 2

The histogram shows the overall spread, the box and violin plots show a clear shift between the classes, and the scatter plot shows that combining it with `worst_radius` separates the two groups even better than either feature alone. Each tool answered a **different question** about the same feature.

---
# Typical Classification EDA Workflow

There is no mandatory order, but a sensible workflow is:

```text
1. Understand the dataset                       (Section 1)
        ↓
2. Understand the target                        (Section 2)
        ↓
3. Check data quality                           (Sections 3, 17)
        ↓
4. Understand individual feature distributions  (Sections 4-8)
        ↓
5. Investigate feature-to-feature relationships (Sections 9-12)
        ↓
6. Investigate feature-to-target relationships  (Section 13)
        ↓
7. Investigate class separation                 (Sections 15, 16)
        ↓
8. Investigate outliers and unusual observations (Section 14)
        ↓
9. Form hypotheses
        ↓
10. Train a baseline model
        ↓
11. Interpret the model                         (Section 18)
        ↓
12. Compare model findings with EDA findings
```

The important part is that **EDA is iterative**:

* You might discover something in a correlation matrix that makes you go back to a scatter plot.
* You might see an unusual cluster in a scatter plot and go back to the raw records.
* You might see class separation in a box plot and then test whether a classifier actually benefits from that feature.

So EDA is less like:

```text
Do 17 plots → Done
```

and more like:

```text
Observe
   ↓
Ask a question
   ↓
Investigate
   ↓
Form a hypothesis
   ↓
Check it using another view of the data
   ↓
Interpret
   ↓
Ask the next question
```

That is the mindset to develop while working through this dataset.

---
# Try it yourself

Use the same tools on questions of your own. Some ideas:

1. Repeat the box-plot / violin-plot comparison for the `*_error` features. Do the standard errors separate the classes as well as the `mean_` and `worst_` versions?
2. Find a feature pair with a correlation around 0.5. Use a scatter plot to check whether a straight line really is a good description.
3. Remove the redundant features (keep one of radius / perimeter / area for mean and worst) and retrain the Random Forest. Does accuracy change? Do the importance rankings change?
4. Use `iqr_outliers` on `worst_concave_points`. Which class do the flagged samples belong to?
5. Run PCA on only the `mean_` features. How much variance do 2 components capture now?